# EN/DE 64k Batch Cosine Similarity

Minimal check: load `data/train_lang.csv` plus the aligned stored BERT embeddings, sample 10 English/German batch pairs, and compute cosine similarity between the two batch mean embeddings.

Two sampling modes are compared:
- `random`: random 64k examples per language
- `balanced`: 64k examples per language, balanced across labels

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

In [ ]:
ROOT = Path("..") if Path("../data/train_lang.csv").exists() else Path(".")
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
EMB_PATH = ROOT / "data" / "bert_embeddings_lang.pt"

BATCH_SIZE = 64_000
NUM_BATCHES = 10
SEED = 42

df = pd.read_csv(TRAIN_CSV)
emb_data = torch.load(EMB_PATH, map_location="cpu")
emb = emb_data["embedding"].float()

assert len(df) == len(emb), (len(df), len(emb))
assert df["id"].to_numpy().tolist() == emb_data["id"].tolist()

print(df.shape)
display(pd.crosstab(df["lang"], df["label"]))

In [ ]:
rng = np.random.default_rng(SEED)
labels = sorted(df["label"].unique())
lang_indices = {lang: df.index[df["lang"].eq(lang)].to_numpy() for lang in ["eng_Latn", "deu_Latn"]}
lang_label_indices = {
    (lang, label): df.index[df["lang"].eq(lang) & df["label"].eq(label)].to_numpy()
    for lang in lang_indices
    for label in labels
}


def sample_random(lang, n=BATCH_SIZE):
    pool = lang_indices[lang]
    return rng.choice(pool, size=n, replace=len(pool) < n)


def sample_balanced(lang, n=BATCH_SIZE):
    base = n // len(labels)
    counts = np.full(len(labels), base)
    counts[: n - base * len(labels)] += 1

    parts = []
    for label, count in zip(labels, counts):
        pool = lang_label_indices[(lang, label)]
        parts.append(rng.choice(pool, size=count, replace=len(pool) < count))

    idx = np.concatenate(parts)
    rng.shuffle(idx)
    return idx


def mean_cosine(en_idx, de_idx):
    en_mean = emb[torch.as_tensor(en_idx)].mean(dim=0)
    de_mean = emb[torch.as_tensor(de_idx)].mean(dim=0)
    return F.cosine_similarity(en_mean, de_mean, dim=0).item()


def run_batches(mode, num_batches=NUM_BATCHES):
    sampler = sample_balanced if mode == "balanced" else sample_random
    rows = []
    for batch in range(1, num_batches + 1):
        en_idx = sampler("eng_Latn")
        de_idx = sampler("deu_Latn")
        rows.append(
            {
                "mode": mode,
                "batch": batch,
                "n_eng": len(en_idx),
                "n_deu": len(de_idx),
                "cosine": mean_cosine(en_idx, de_idx),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
results = pd.concat(
    [run_batches("random"), run_batches("balanced")],
    ignore_index=True,
)

display(results)
display(results.groupby("mode")["cosine"].agg(["mean", "std", "min", "max"]))